# ◆ LUMEN Protocol — full stack in Colab (GPU-ready)

**MVM (Rust) + PDB (hierarchical memory) + M-Light + Astrid** — the open metal of an AI agent society, in your browser. MIT licensed.

What this notebook does, step by step:

1. Clones [lumen-protocol](https://github.com/GonzaloMonzonC/lumen-protocol) and [Astrid](https://github.com/GonzaloMonzonC/astrid) (both MIT).
2. Builds (or downloads) the Rust **MVM** as a shared library for Linux.
3. Creates a throwaway **PDB** and seeds it with demo state.
4. Runs **Astrid's evidence digest** (`EVIDENCE^ASTRID`) — the agent that refuses to speculate: every claim cites its source global.
5. Shows **stability + anchoring**: same state → same digest → same `cid` (sha256). That is the notary contract.
6. **GPU vitamins** (optional): semantic search over the digest claims with local embeddings — accelerated when a GPU runtime is attached.

> Runtime: **Runtime ▸ Change runtime type ▸ T4 GPU** recommended (steps 1–5 work on CPU too).


In [ ]:
# 1. Environment + clone (idempotent)
import os, sys, time, platform, subprocess, shutil
REPO = "/content/lumen-protocol"
ASTRID = "/content/astrid"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/GonzaloMonzonC/lumen-protocol.git", REPO], check=True)
if not os.path.exists(ASTRID):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/GonzaloMonzonC/astrid.git", ASTRID], check=True)
print("python:", platform.python_version())
print("cargo:", shutil.which("cargo") or "NOT INSTALLED (will install in step 2)")
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
    print("GPU:", gpu.stdout.strip() or "none")
except Exception:
    print("GPU: none detected")


In [ ]:
# 2. Rust toolchain (only needed to build the MVM)
# The wrapper (lumen_mlight.py) builds the release cdylib automatically when missing.
# First build takes ~2-4 min on Colab; subsequent runs reuse the compiled lib.
if not shutil.which("cargo"):
    print("Installing rustup...")
    subprocess.run(["curl", "--proto", "=https", "--tlsv1.2", "-sSf",
                    "https://sh.rustup.rs"], capture_output=True)
    # rustup needs a shell env; simplest reliable path on Colab:
    subprocess.run(["bash", "-c",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"],
        check=True)
    os.environ["PATH"] = os.environ.get("PATH", "") + ":" + os.path.expanduser("~/.cargo/bin")
import importlib.util
sys.path.insert(0, REPO + "/implementations/mcp-servers/pdb")
import lumen_mlight as lm
print("wrapper loaded; lib target:", lm._lib_path())
# Run exprés: si hay Release con el .so precompilado (Linux), bájalo a la
# ruta exacta que espera el wrapper — ensure_built() lo detecta y SALTA la
# compilación de 2-4 min. Sin Release (aún), compila normal.
import urllib.request
try:
    _so_dir = os.path.dirname(lm._lib_path())
    os.makedirs(_so_dir, exist_ok=True)
    _so_path = os.path.join(_so_dir, os.path.basename(lm._lib_path()))
    if not os.path.exists(_so_path):
        _urls = [
            "https://github.com/GonzaloMonzonC/lumen-protocol/releases/latest/download/liblumen_mlight.so",
            "https://github.com/GonzaloMonzonC/lumen-protocol/releases/download/mlight-v0.1.0/liblumen_mlight.so",
        ]
        _ok = False
        for _u in _urls:
            try:
                urllib.request.urlretrieve(_u, _so_path)
                _ok = True
                print("descargado .so precompilado:", os.path.getsize(_so_path), "bytes")
                break
            except Exception:
                continue
        if not _ok:
            print("sin .so precompilado (release no accesible) — se compilara")
    else:
        print(".so ya presente en target/release")
except Exception as e:
    print("sin .so precompilado (normal si aun no hay Release):", e)
t0 = time.time()
ok = lm.ensure_built()
print("ensure_built:", ok, f"({time.time()-t0:.0f}s)")
assert ok, "MVM build failed — see output above"
print("MVM available:", lm.available())


In [ ]:
# 3. Throwaway PDB + seed demo state (M code, real MVM)
DB = "/content/lumen_demo.db"
if os.path.exists(DB):
    os.remove(DB)

seed = """
S ^ACTIVE="astrid-demo"
S ^PERSONALITY("astrid","version")="0.3.0"
S ^PERSONALITY("astrid","is_active")="1"
S ^PERSONALITY("astrid","evidence_routine")="EVIDENCE^ASTRID"
S ^ANGI("metrics","agents_online")="{\"value\": 3, \"updated\": \"2026-09-09T12:00:00Z\"}"
S ^ANGI("metrics","alerts_last_run")="0"
S ^SYS("MCP","worker1","type")="http"
S ^SYS("MCP","worker1","url")="https://worker1.example/mcp"
S ^SYS("MCP","worker2","type")="http"
S ^SYS("MCP","worker2","url")="https://worker2.example/mcp"
S ^QUANTUM("colapso",1)="{\"idx\": 1, \"backend\": \"sim\"}"
S ^QUANTUM("colapso",2)="{\"idx\": 2, \"backend\": \"sim\"}"
W "seed ok"
"""
r = lm.execute(seed, routines={}, sqlite_path=DB, gas_limit=50000)
r2 = lm.execute('W "active=",$G(^ACTIVE)," workers=",$O(^SYS("MCP",""))', routines={}, sqlite_path=DB, gas_limit=5000)
out2 = (r2.get("state") or {}).get("output", "")
print("seed output:", (r.get("state") or {}).get("output", "") or "(ok via check)")
print("check:", out2)
assert "astrid-demo" in str(out2), "seed no persistio"


In [ ]:
# 4. Astrid's evidence digest — every claim cites its source
astrid_src = open(ASTRID + "/src/astrid.m", encoding="utf-8").read()
out = lm.execute("W $$EVIDENCE^ASTRID()", routines={"ASTRID": astrid_src},
                 sqlite_path=DB, gas_limit=80000)
digest = ((out.get("state") or {}).get("output") or "") if isinstance(out, dict) else ""
print(digest)
assert "evidence=true" in digest, "canary failed"
claims = [ln for ln in digest.split("\n") if ln.startswith("claim|")]
print(f"--- {len(claims)} claims, all with a visible source ---")


In [ ]:
# 5. Notary contract: stability + content address
# Same state -> same digest -> same cid. A verifier (any agent with the key) can
# confirm who produced a digest and when — see Astrid docs/EVIDENCE_SCHEMA.md §3.
import hashlib, hmac
out2 = lm.execute("W $$EVIDENCE^ASTRID()", routines={"ASTRID": astrid_src},
                  sqlite_path=DB, gas_limit=80000)
d2 = ((out2.get("state") or {}).get("output") or "") if isinstance(out2, dict) else ""
cid1, cid2 = hashlib.sha256(digest.encode()).hexdigest(), hashlib.sha256(d2.encode()).hexdigest()
print("stable digest:", digest == d2)
print("cid:", cid1[:24] + "…")
assert digest == d2 and cid1 == cid2

# anchor the digest into the PDB ledger (demo key; the real runtime signs with
# ^CONFIG("ddp_hmac_key") — the contract is what matters here)
demo_key = "colab-demo-key"
ts = str(int(time.time()))
sig = hmac.new(demo_key.encode(), (ts + cid1 + demo_key).encode(), hashlib.sha256).hexdigest()
lines = [ln for ln in digest.split("\n") if ln]
sets = ['S ^EVIDENCE("' + cid1 + '")="' + sig + "|" + ts + '|EVIDENCE^ASTRID"']
for i, ln in enumerate(lines, start=1):
    sets.append('S ^EVIDENCE("' + cid1 + '","digest",' + str(i) + ')="' + ln.replace('"', '""') + '"')
r3 = lm.execute("\n".join(sets), routines={}, sqlite_path=DB, gas_limit=100000)
chk = lm.execute('W $D(^EVIDENCE("' + cid1 + '"))', routines={}, sqlite_path=DB, gas_limit=5000)
print("ledger ^EVIDENCE(cid) present:", "11" in ((chk.get("state") or {}).get("output") or ""))
print("anchored:", len(lines), "digest lines under ^EVIDENCE(cid,'digest',n)")


In [ ]:
# 6. GPU vitamins — semantic search over the digest claims
# Local embeddings (fastembed, bge-small). With a GPU runtime Colab uses
# onnxruntime-gpu; on CPU this still runs (small corpus, quick).
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "fastembed", "onnxruntime", "numpy"], check=True, timeout=300)
except Exception as e:
    print("pip install failed:", e)
try:
    t0 = time.time()
    from fastembed import TextEmbedding
    model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
    print(f"embedding model ready ({time.time()-t0:.0f}s)")
    docs = claims + ["REGLA: responde solo con datos registrados"]
    vecs = list(model.embed(docs))
    print("embedded", len(vecs), "lines, dim:", len(vecs[0]), f"({time.time()-t0:.0f}s total)")

    import numpy as np
    Q = "que workers MCP hay registrados?"
    qv = list(model.embed([Q]))[0]
    scores = [float(np.dot(qv, v) / (np.linalg.norm(qv) * np.linalg.norm(v))) for v in vecs]
    top = sorted(zip(scores, docs), key=lambda x: -x[0])[:3]
    print("query:", Q)
    for s, d in top:
        print(f"  {s:.3f}  {d[:90]}")
except Exception as e:
    print("GPU/semantic step skipped (optional):", type(e).__name__, str(e)[:120])


## What you just ran

- **MVM**: the Rust virtual machine executed real M code (seeding, digest, ledger writes).
- **PDB**: hierarchical, transactional memory (SQLite-backed) — the same substrate agents share.
- **M-Light**: the M evaluator, portable and deterministic.
- **Astrid**: her digest emitted claims with visible sources (`claim|kind|source|value|d`) — no source, no claim. Same state → same `cid`: that is the seed of the notary contract (content-addressed, signed digests — see `astrid/docs/EVIDENCE_SCHEMA.md`).

Next steps: build your own agent from [Astrid's template](https://github.com/GonzaloMonzonC/astrid/tree/main/template), run the [full test suite](https://github.com/GonzaloMonzonC/astrid) (20 checks against a real MVM), or explore the [protocol docs](https://github.com/GonzaloMonzonC/lumen-protocol/tree/main/docs).

*Demo data is fictional — the notebook is fully self-contained and MIT-clean.*
